In [ ]:
import sys
import numpy as np
import pandas as pd
from collections import defaultdict
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import RegressionResultsWrapper 

sys.path.append('..')
from modules.source import Sources, parquet_monthly

CHANNELS: dict[str, tuple[str]] = {
  "Domestic Baseline": (Sources.cpi_inflation, Sources.IPI),
  "Rate Differential": (Sources.rate_differential,),
  "Global Risk Appetite": (Sources.VIX,),
  "Global Credit Condition": (Sources.UST_10Y, Sources.HY_OAS),
  "FX Pass-Through": (Sources.USDMYR,),
  "Commodity": (Sources.crude_palm_oil, Sources.brent_oil),
}
ALL_PREDICTORS = [v for vlist in CHANNELS.values() for v in vlist]
DEPENDENT_VARS = [Sources.KLCI, Sources.financials, Sources.energy, Sources.industrial_products, Sources.plantation, Sources.REITs, Sources.technology]

df_monthly = pd.read_parquet(parquet_monthly).xs("change", axis=1, level=1)

def fit_hac_model(target, rhs_vars, data) -> RegressionResultsWrapper:
  formula = f"{target} ~ {' + '.join(rhs_vars)}"
  L = int(np.floor(4 * (len(data) / 100) ** (2 / 9)))
  return smf.ols(formula, data=data).fit(cov_type="HAC", cov_kwds={"maxlags": L})

full_models, fit_dict, partialr2_dict = {}, {}, {}
wald_dict = defaultdict(dict)

for d_var in DEPENDENT_VARS:
  data = df_monthly[[d_var] + ALL_PREDICTORS].dropna()
  m_full = fit_hac_model(d_var, ALL_PREDICTORS, data)
  full_models[d_var] = m_full

  fit_dict[d_var] = {
    ("Full R2"): round(m_full.rsquared, 4),
    ("Adj. R2"): round(m_full.rsquared_adj, 4),
  }

  partialr2_dict[d_var] = {}
  for ch_name, ch_vars in CHANNELS.items():
    for var in ch_vars:
      rhs_drop = [x for x in ALL_PREDICTORS if x != var] # exclude the current predictor 
      m_drop = fit_hac_model(d_var, rhs_drop, data)
      partialr2_dict[d_var][(ch_name, var)] = round(m_full.rsquared - m_drop.rsquared, 4)

  for ch_name, ch_vars in CHANNELS.items():
    hypothesis = ", ".join(f"{v} = 0" for v in ch_vars)
    w_res = m_full.wald_test(hypothesis, use_f=True, scalar=True)

    wald_dict[ch_name][(d_var, "F-Stat")] = round(float(w_res.statistic), 4)
    wald_dict[ch_name][(d_var, "p-value")] = round(float(w_res.pvalue), 4)

df_fit_matrix = pd.DataFrame(fit_dict)
df_wald_matrix = pd.DataFrame(wald_dict).T
df_partialr2 = pd.DataFrame(partialr2_dict)

### 1. ADF (Stationary), VIF (Multicollinearity)

In [27]:
print("Diagnostic Assumptions (ADF, VIF):")

diagnostic_dict = {}
for col in ALL_PREDICTORS + DEPENDENT_VARS:
  stat, p = adfuller(df_monthly[col].dropna(), autolag="AIC", result_object=True)[:2]
  diagnostic_dict[col] = {
    ("ADF", "score"): round(float(stat), 4),
    ("ADF", "p-val"): round(float(p), 4)
  }

exog_matrix = full_models[DEPENDENT_VARS[0]].model.exog
exog_names = full_models[DEPENDENT_VARS[0]].model.exog_names
for i, var_name in enumerate(exog_names):
  if var_name != "Intercept":
    vif_score = variance_inflation_factor(exog_matrix, i)
    diagnostic_dict[var_name][("VIF", "score")] = round(float(vif_score), 4)

diagnostic_dict = pd.DataFrame(diagnostic_dict).T
diagnostic_dict.columns.names = ["Test", ""]
print(diagnostic_dict)

Diagnostic Assumptions (ADF, VIF):
Test                     ADF             VIF
                       score   p-val   score
UST_10Y             -11.1366  0.0000  1.3136
HY_OAS               -9.3741  0.0000  2.5676
FFR_OPR_diff         -4.7400  0.0001  1.1928
VIX                  -5.7063  0.0000  1.6315
USDMYR              -11.0840  0.0000  1.2665
CPO                  -9.1417  0.0000  1.0883
Oil                  -8.5630  0.0000  1.9657
cpi_inflation        -9.3565  0.0000  1.2350
IPI                  -4.5511  0.0002  1.2611
KLCI                -13.1469  0.0000     NaN
financials          -10.4197  0.0000     NaN
energy              -11.0151  0.0000     NaN
industrial_products -10.0120  0.0000     NaN
plantation          -13.8698  0.0000     NaN
REITs               -14.8697  0.0000     NaN
technology           -4.6504  0.0001     NaN


### 4. MODEL FIT & VARIANCE DECOMPOSITION

In [30]:
print("Model Fit:")
print(df_fit_matrix.to_string())
print("\nChannel Variance Decomposition (Contribution to R²):")
print(df_partialr2.groupby(level=0, sort=False).sum().to_string())

Model Fit:
           KLCI  financials  energy  industrial_products  plantation   REITs  technology
Full R2  0.3190      0.2651  0.2368               0.1496      0.1201  0.1922      0.2338
Adj. R2  0.2662      0.2081  0.1775               0.0837      0.0367  0.1296      0.1744

Channel Variance Decomposition (Contribution to R²):
                           KLCI  financials  energy  industrial_products  plantation   REITs  technology
Global Credit Condition  0.0478      0.0306  0.0163               0.0618      0.0003  0.0630      0.0796
Rate Differential        0.0054      0.0002  0.0025               0.0156      0.0350  0.0176      0.0000
Global Risk Appetite     0.0053      0.0087  0.0081               0.0012      0.0000  0.0015      0.0206
FX Pass-Through          0.0443      0.0637  0.0003               0.0002      0.0005  0.0004      0.0099
Commodity                0.0144      0.0060  0.1221               0.0069      0.0553  0.0104      0.0183
Domestic Baseline        0.0074      0

### 5. INFERENCE & FEATURE IMPORTANCE

In [29]:
print("Channel Wald Joint Significance Tests:")
print(df_wald_matrix.to_string())

Channel Wald Joint Significance Tests:
                            KLCI         financials          energy         industrial_products         plantation           REITs         technology        
                          F-Stat p-value     F-Stat p-value  F-Stat p-value              F-Stat p-value     F-Stat p-value  F-Stat p-value     F-Stat p-value
Global Credit Condition   4.9985  0.0083     4.0893  0.0192  0.8500  0.4301              3.0552  0.0509     0.0145  0.9856  3.1761  0.0454     6.4062  0.0023
Rate Differential         0.7586  0.3856     0.0541  0.8164  0.3935  0.5317              1.5535  0.2151     2.8585  0.0942  2.3233  0.1302     0.0052  0.9428
Global Risk Appetite      1.0185  0.3150     0.9171  0.3402  0.9913  0.3215              0.1941  0.6603     0.0016  0.9682  0.1501  0.6992     1.9110  0.1695
FX Pass-Through          10.4073  0.0016    11.1200  0.0011  0.0434  0.8354              0.0373  0.8472     0.0723  0.7886  0.0333  0.8556     1.4758  0.2269
Commodity    